# Week 39, Day 2: Review: types and design

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w039d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from typing import NamedTuple

class Size(Enum):
    SMALL = 1
    LARGE = 2

class Point(NamedTuple):
    x: int
    y: int = 0

@dataclass(order=True, frozen=True)
class Order:
    total: int
    size: Size = field(compare=False, default=Size.SMALL)

print(Size(2), Size["SMALL"].value, Point(3), Point(3)._replace(y=4))
print(sorted([Order(5, Size.LARGE), Order(2)]), Order(2) == Order(2, Size.LARGE))

In [ ]:
from abc import ABC, abstractmethod
from typing import Protocol, runtime_checkable

class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

    def describe(self) -> str:                 # shared code, using the abstract method
        return f"{type(self).__name__} of area {self.area():g}"

class Square(Shape):
    def __init__(self, side: float) -> None:
        self.side = side

    def area(self) -> float:
        return self.side ** 2

@runtime_checkable
class Closable(Protocol):
    def close(self) -> None: ...

class Door:
    def close(self) -> None:
        print("door closed")

print(Square(3).describe())
try:
    Shape()
except TypeError as err:
    print("TypeError:", err)
print(isinstance(Door(), Closable), isinstance(Square(1), Closable))

In [ ]:
from typing import Callable

def best[T](items: list[T], score: Callable[[T], float]) -> T | None:
    """The item with the highest score, or None for an empty list."""
    if not items:
        return None
    return max(items, key=score)

word = best(["fig", "banana", "kiwi"], len)
print(word, best([], len))
if word is not None:              # mypy insists on this check before .upper()
    print(word.upper())

**Predict before you run:** what will this print?



In [ ]:
def price(kind, base):
    if kind == "child":
        return base * 0.5
    elif kind == "senior":
        return base * 0.7
    elif kind == "adult":
        return base
    return base

RATES = {"child": 0.5, "senior": 0.7}

def price2(kind, base):
    return base * RATES.get(kind, 1)

print([price(k, 10) for k in ["child", "senior", "adult", "student"]])
print([price2(k, 10) for k in ["child", "senior", "adult", "student"]])

**Predict before you run:** what will this print?



In [ ]:
class AppError(Exception):
    pass

class Missing(AppError, LookupError):
    pass

def find(d, key):
    try:
        return d[key]
    except KeyError as err:
        raise Missing(f"no {key!r}") from err

log = []
for key in ["a", "b"]:
    try:
        log.append(find({"a": 1}, key))
    except LookupError as err:
        log.append((type(err).__name__, str(err), type(err.__cause__).__name__))
    else:
        log.append("found")
    finally:
        log.append("|")
print(log)

> **Key idea.** Dataclasses for records, Enums for fixed choices, NamedTuples for small immutable tuples, ABCs for families that must implement methods, Protocols for "anything with these methods". Hints describe types (`int | None`, `list[T]`, `Callable`) and mypy checks them. Design for cohesion and low coupling; SOLID, DRY, KISS and YAGNI name the usual problems; refactor in small tested steps. Catch narrowly, chain with `from`, log once where handled, `raise` for callers and `assert` for your own bugs.

### Exercise: A checked, ordered dataclass

Write a frozen, ordered dataclass `Parcel` with `weight_g` (int), `dest` (str) and `tags` (a tuple of strings, default empty). Parcels sort by weight, then destination; `tags` takes no part in comparisons. `__post_init__` raises `ValueError` for a weight of 0 or less, or an empty destination. Also write a `Speed` Enum with `STANDARD = "std"` and `EXPRESS = "exp"`, and `cost(parcel, speed)`: 1 penny per 10 g or part of 10 g (so 1 to 10 g costs 1), doubled for `EXPRESS`.

In [ ]:
from dataclasses import dataclass, field
from enum import Enum

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import dataclasses\ndef test_parcel():\n    \"the dataclass\"\n    a, b = Parcel(500, \"Leeds\"), Parcel(500, \"Abuja\", (\"fragile\",))\n    assert sorted([a, b]) == [b, a] and Parcel(5, \"x\", (\"a\",)) == Parcel(5, \"x\", (\"b\",))\n    assert len({a, Parcel(500, \"Leeds\", (\"z\",))}) == 1\n    try:\n        a.weight_g = 1\n        assert False, \"frozen\"\n    except dataclasses.FrozenInstanceError:\n        pass\n    for args in [(0, \"x\"), (10, \"\")]:\n        try:\n            Parcel(*args)\n            assert False\n        except ValueError:\n            pass\ndef test_cost():\n    \"costs\"\n    assert Speed(\"exp\") is Speed.EXPRESS and Speed[\"STANDARD\"].value == \"std\"\n    assert cost(Parcel(1, \"x\"), Speed.STANDARD) == 1 and cost(Parcel(10, \"x\"), Speed.STANDARD) == 1 and cost(Parcel(11, \"x\"), Speed.STANDARD) == 2\n    assert cost(Parcel(500, \"x\"), Speed.EXPRESS) == 100")

### Exercise: An ABC and a Protocol

Write an abstract base class `Shape` with an abstract `area()` and a concrete `bigger_than(other)` (true if this shape's area is larger than `other`'s). Write `Rect(w, h)` and `Circle(r)` (use `math.pi`) as subclasses. Then a `runtime_checkable` Protocol `HasArea` (just `area()`), and `total_area(things)`, which adds up the areas of the items that fit `HasArea` and skips the rest.

In [ ]:
import math
from abc import ABC, abstractmethod
from typing import Protocol, runtime_checkable

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_shapes():\n    \"the family\"\n    try:\n        Shape()\n        assert False, \"Shape is abstract\"\n    except TypeError:\n        pass\n    assert Rect(2, 3).area() == 6 and round(Circle(1).area(), 4) == 3.1416\n    assert Rect(2, 3).bigger_than(Circle(1)) and not Circle(1).bigger_than(Rect(2, 3))\n    class Half(Shape):\n        pass\n    try:\n        Half()\n        assert False, \"a subclass without area can't be made\"\n    except TypeError:\n        pass\ndef test_protocol():\n    \"the protocol\"\n    class Field:\n        def area(self):\n            return 100\n    assert isinstance(Field(), HasArea) and not isinstance(\"x\", HasArea)\n    assert total_area([Rect(1, 2), Field(), \"label\", 5]) == 102")

### Exercise: Hints that tell the truth

Add type hints to the three functions so that they describe exactly what they take and return: `average` takes a list of floats and returns a float or `None`; `group` takes a list of strings and a function from string to string, and returns a dictionary from string to list of strings; `pick` is **generic**: it takes a list of any type `T` and an int, and returns a `T`. Don't change the bodies.

In [ ]:
from typing import Callable

def average(values):
    return sum(values) / len(values) if values else None

def group(words, key):
    out = {}
    for w in words:
        out.setdefault(key(w), []).append(w)
    return out

def pick(items, i):
    return items[i]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import typing\ndef test_hints():\n    \"the hints\"\n    h = typing.get_type_hints(average)\n    assert h[\"values\"] == list[float] and h[\"return\"] in (float | None, typing.Optional[float])\n    h = typing.get_type_hints(group)\n    assert h[\"words\"] == list[str] and h[\"key\"] == typing.Callable[[str], str] and h[\"return\"] == dict[str, list[str]]\n    params = getattr(pick, \"__type_params__\", ())\n    assert len(params) == 1, \"write def pick[T](...)\"\n    T = params[0]\n    a = pick.__annotations__\n    assert a[\"items\"] == list[T] and a[\"i\"] is int and a[\"return\"] is T")

### Exercise: Refactor a conditional

`shipping` works, but every new country means another `elif` in the middle of it. Refactor it into a table: `RATES`, a dictionary from country to `(base, per_kg)`, and a much shorter `shipping` that looks the country up, keeping **exactly** the same results, including the `ValueError` for an unknown country (message `"no rates for 'XX'"`). No `if country ==` may remain.

In [ ]:
def shipping(country, kg):
    if country == "UK":
        base, per_kg = 300, 50
    elif country == "NG":
        base, per_kg = 900, 200
    elif country == "FR":
        base, per_kg = 600, 120
    else:
        raise ValueError(f"no rates for {country!r}")
    if kg <= 0:
        raise ValueError("weight must be positive")
    return base + per_kg * kg

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_same():\n    \"same results\"\n    assert [shipping(c, k) for c, k in [(\"UK\", 1), (\"NG\", 2.5), (\"FR\", 3)]] == [350, 1400.0, 960]\n    for args, message in [((\"XX\", 1), \"no rates for 'XX'\"), ((\"UK\", 0), \"weight must be positive\")]:\n        try:\n            shipping(*args)\n            assert False\n        except ValueError as err:\n            assert str(err) == message\ndef test_table():\n    \"a table\"\n    assert RATES[\"NG\"] == (900, 200) and len(RATES) == 3\n    assert \"country ==\" not in source and \"elif\" not in source")

### Exercise: Robust reading

Write `ConfigError(Exception)` and `read_numbers(lines, log)`: each line should be a number. Return the list of floats. A line that isn't a number is skipped, with a `WARNING` logged: `"line N: not a number: 'TEXT'"` (N from 1). If **more than half** of the lines (not counting blank lines, which are skipped silently) were bad, raise `ConfigError("too many bad lines: B of T")`. Catch only `ValueError`.

In [ ]:
def read_numbers(lines, log):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import io, logging\ndef fresh():\n    s = io.StringIO()\n    log = logging.getLogger(\"review.robust\")\n    log.handlers.clear()\n    h = logging.StreamHandler(s)\n    h.setFormatter(logging.Formatter(\"%(levelname)s %(message)s\"))\n    log.addHandler(h)\n    log.setLevel(logging.DEBUG)\n    log.propagate = False\n    return log, s\ndef test_read():\n    \"reading\"\n    log, s = fresh()\n    assert read_numbers([\"1.5\", \"\", \"x\", \" 2 \", \"3\"], log) == [1.5, 2.0, 3.0]\n    assert s.getvalue() == \"WARNING line 3: not a number: 'x'\\n\", repr(s.getvalue())\ndef test_too_many():\n    \"too many\"\n    log, s = fresh()\n    try:\n        read_numbers([\"a\", \"1\", \"b\"], log)\n        assert False\n    except ConfigError as err:\n        assert str(err) == \"too many bad lines: 2 of 3\"\n    log, s = fresh()\n    assert read_numbers([\"a\", \"1\"], log) == [1.0], \"exactly half is fine\"\n    try:\n        read_numbers([None], log)\n        assert False, \"None is a bug: TypeError\"\n    except TypeError:\n        pass")

### Bug hunt: A shared list of tags

This "dataclass" was written as a plain class with a list default in `__init__`, so every note without tags shares one list, and tagging one tags them all: this prints `['urgent'] ['urgent']`. Make it a proper dataclass with its own list for each note, so it prints `['urgent'] []`.

In [ ]:
class Note:
    def __init__(self, text, tags=[]):
        self.text = text
        self.tags = tags

a, b = Note("call Ada"), Note("buy tea")
a.tags.append("urgent")
print(a.tags, b.tags)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a list each\"\n    expect_output(\"['urgent'] []\")\n    import dataclasses\n    assert dataclasses.is_dataclass(Note) and Note(\"x\") == Note(\"x\")")

### Bug hunt: Maybe None

`find_user` returns `None` when there's no match, and mypy would say `Item "None" of "dict[str, str] | None" has no attribute "get"`: here it crashes. Handle the `None` case so this prints `Ada Obi` then `unknown user`.

In [ ]:
USERS = [{"id": "u1", "name": "Ada Obi"}]

def find_user(uid: str) -> dict[str, str] | None:
    for u in USERS:
        if u["id"] == uid:
            return u
    return None

def name_of(uid: str) -> str:
    user = find_user(uid)
    return user.get("name", "")

print(name_of("u1"))
print(name_of("u9"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"check for None\"\n    expect_output(\"Ada Obi\\nunknown user\")")

### Bug hunt: A square that breaks its promise

`Square` inherits from `Rectangle` but changes both sides in `set_width`, so code that trusts a `Rectangle` gets a wrong answer (Liskov). Here `stretch` expects the area to become 20 but gets 25 for the square. Fix the design: make `Square` **not** a `Rectangle` (give it its own class with `side`, `set_side` and `area`), so the program prints `20` then `Square 25`.

In [ ]:
class Rectangle:
    def __init__(self, w, h):
        self.w, self.h = w, h

    def set_width(self, w):
        self.w = w

    def area(self):
        return self.w * self.h

class Square(Rectangle):
    def __init__(self, side):
        super().__init__(side, side)

    def set_width(self, w):
        self.w = self.h = w

def stretch(rect):
    rect.set_width(5)
    return rect.area()

print(stretch(Rectangle(2, 4)))
s = Square(2)
print(type(s).__name__, stretch(s))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"no broken promise\"\n    assert not issubclass(Square, Rectangle)\n    s = Square(3)\n    s.set_side(5)\n    assert s.area() == 25 and s.side == 5\n    assert output.splitlines()[0] == \"20\"")

### Puzzle: Translating an error

Arrange the lines into a function that turns a missing key into the program's own error, keeping the original as the cause, and logs nothing (the caller will). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
from dataclasses import dataclass, field
from enum import Enum, auto

class Colour(Enum):
    RED = auto()
    GREEN = auto()

@dataclass
class Item:
    name: str
    tags: list = field(default_factory=list)

a = Item("x")
b = Item("x")

p = ___   # Colour.GREEN.value
q = ___   # a == b
r = ___   # a is b
s = ___   # Colour(1).name
t = ___   # len({Colour.RED, Colour(1), Colour.GREEN})

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 2 and q is True and r is False, \"p, q and r\"\n    assert s == \"RED\" and t == 2, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a typed event schedule

Build a small, typed, robust schedule:

- `Kind`, an Enum with `TALK = "talk"`, `WORKSHOP = "workshop"`, `BREAK = "break"`.
- `Slot`, a frozen, ordered dataclass with `start` and `end` (minutes since midnight, ints), `title` (str) and `kind` (a `Kind`), ordered by start time then end. `__post_init__` raises `ValueError` if `end <= start` or the title is blank.
- `ScheduleError(Exception)` and `Clash(ScheduleError)`, which stores `first` and `second` (the two slots) and has the message `"'A' clashes with 'B'"` (the titles, earlier slot first).
- `parse_slot(line: str) -> Slot` reads `"09:30-10:15 talk Opening remarks"`: times as `HH:MM`, then the kind's value, then the title (the rest). Any problem (bad time, unknown kind, no title, the dataclass's own check) raises `ScheduleError(f"bad slot: {line!r}")` chained **from** the original error.
- `build(lines: list[str]) -> list[Slot]` parses every non-blank line, sorts them, and raises `Clash` for the first pair of neighbours that overlap. Breaks may touch other slots, but nothing may overlap.

Give every function full type hints.

In [ ]:
from dataclasses import dataclass
from enum import Enum

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import typing\ndef test_parse():\n    \"parsing\"\n    s = parse_slot(\"09:30-10:15 talk Opening remarks\")\n    assert s == Slot(570, 615, \"Opening remarks\", Kind.TALK) and s.kind is Kind.TALK\n    for line in [\"9h30-10:00 talk X\", \"09:30-10:00 lecture X\", \"09:30-10:00 talk\", \"10:00-09:30 talk X\", \"09:30 talk X\", \"09:30-10:70 talk X\"]:\n        try:\n            parse_slot(line)\n            assert False, f\"{line!r} should raise ScheduleError\"\n        except ScheduleError as err:\n            assert str(err) == f\"bad slot: {line!r}\" and err.__cause__ is not None, f\"{line!r}: chain it from the original\"\ndef test_build():\n    \"building\"\n    lines = [\"11:00-12:30 workshop Testing\", \"\", \"09:30-10:15 talk Opening\", \"10:15-11:00 break Tea\"]\n    assert [s.title for s in build(lines)] == [\"Opening\", \"Tea\", \"Testing\"]\n    try:\n        build([\"09:00-10:00 talk A\", \"09:45-10:30 workshop B\", \"12:00-13:00 talk C\"])\n        assert False\n    except Clash as err:\n        assert str(err) == \"'A' clashes with 'B'\" and err.first.title == \"A\" and err.second.title == \"B\"\n        assert isinstance(err, ScheduleError)\ndef test_hints():\n    \"type hints\"\n    assert typing.get_type_hints(parse_slot) == {\"line\": str, \"return\": Slot}\n    assert typing.get_type_hints(build) == {\"lines\": list[str], \"return\": list[Slot]}")

## Warm-up

Four of Week 32's tools in a few lines. `compare=False` leaves `size` out of `==` and ordering, so two orders with the same total are equal whatever their size.

## Describing data

| Tool | Use it for | Notes |
|---|---|---|
| `@dataclass` | records with behaviour: mutable by default | writes `__init__`, `__repr__`, `__eq__`; `order=True`, `frozen=True` (then hashable), `field(default_factory=list)`, `__post_init__` for checks |
| `Enum` | a fixed set of named choices | members are singletons: compare with `is` or `==`; `Size(2)` by value, `Size["SMALL"]` by name; `auto()`, `StrEnum`, `IntEnum`, `Flag` |
| `NamedTuple` | small immutable records that are also tuples | unpack, index, `_replace`, `_asdict`; hashable if the fields are |
| ABC (`abc.ABC` + `@abstractmethod`) | a family that must provide certain methods, checked when an object is **made** | can share code with subclasses |
| `Protocol` | "anything with these methods", checked by **mypy** (and by `isinstance` with `@runtime_checkable`) | no inheritance needed: structural typing, duck typing with a contract |

A `Door` never mentions `Closable`, yet it counts as one, because it has the method. That's the difference: an ABC is a family you **join**; a Protocol is a shape you **fit**.

## Type hints

Hints describe what a function takes and returns. Python ignores them when running; tools like mypy (Week 33) read them and find mistakes before the program runs.

| Hint | Means |
|---|---|
| `int`, `str`, `list[int]`, `dict[str, float]`, `tuple[int, str]`, `tuple[int, ...]` | the type, and the types inside |
| `int \| None` (or `Optional[int]`) | an `int` or `None`: you must check before using it as an int |
| `int \| str` | either |
| `Callable[[int, int], bool]` | a function taking two ints and returning a bool |
| `def first[T](xs: list[T]) -> T` | a generic function: the return type follows the argument's (Python 3.12+) |
| `Iterable[str]`, `Sequence[str]`, `Mapping[str, int]` | accept the widest thing that works (from `collections.abc`) |

Reading mypy's output: it names the file, the line, the problem and a code in brackets.

```text
shop.py:4: error: Item "None" of "Match[str] | None" has no attribute "group"  [union-attr]
shop.py:11: error: "Account" has no attribute "ballance"; maybe "balance"?  [attr-defined]
shop.py:22: error: Missing positional argument "text" in call to "send"  [call-arg]
```

The first is the most common real bug mypy finds: using something that might be `None` as if it never is.

## Design principles

| Principle | In one line | A smell that breaks it |
|---|---|---|
| **Cohesion** | each piece does one job | a function that reads, calculates and prints |
| **Low coupling** | pieces depend on little about each other | reaching through objects: `order.customer.address.city` |
| **S**ingle responsibility | one reason to change | a `Report` class that also sends email |
| **O**pen/closed | add behaviour by adding code, not editing it | a growing `if kind == ...` chain |
| **L**iskov substitution | a subclass works wherever its parent does | a `Square(Rectangle)` whose `set_width` also changes the height |
| **I**nterface segregation | small interfaces | a `Machine` protocol forcing printers to `fax` |
| **D**ependency inversion | depend on abstractions, pass details in | `self.clock = SystemClock()` inside a class |
| **DRY** | one place for each piece of knowledge | the same rule written in three functions |
| **KISS** | the simplest design that does the job | a plugin system for two cases |
| **YAGNI** | don't build what nobody needs yet | options no caller passes |

**Refactoring** (Week 34) changes structure without changing behaviour, in small steps with the tests run after each: rename, extract a function, replace a conditional with polymorphism or a table, introduce a parameter object.

The same behaviour, refactored from an `if` chain into a table: adding a student discount is now one dictionary entry (open/closed), and the rule lives in one place (DRY).

## Robust programs

Week 37 in brief:

- **The exception tree**: `BaseException` at the top; `KeyboardInterrupt` and `SystemExit` outside `Exception`. Catch the narrowest family you can handle, specific clauses first; never a bare `except:`.
- **`try`/`except`/`else`/`finally`**: `else` runs only if nothing was raised (keep `try` small); `finally` always runs; never `return` from `finally`.
- **Your own family**: one base class per program, subclasses for what callers handle differently, also inheriting from a built-in when it really is one.
- **Chaining**: `raise New(...) from err` (deliberate translation), a plain raise in `except` ("during handling", often an accident), `from None` (hide). `add_note` adds context; a bare `raise` re-raises.
- **EAFP vs LBYL**: let operations decide their own rules (`int(text)`, `open(path, "x")`); check first for your own rules and before changes that mustn't stop halfway.
- **Logging**: levels, `getLogger(__name__)`, handlers and formatters, `log.exception` in `except`; log an error once, where it's handled; libraries never configure logging.
- **Defensive programming**: `raise` for anything a caller can cause, `assert` only for your own bugs (`-O` removes asserts); fail fast; defensive copies.

## The classic mistakes

1. **`@dataclass` with a list default** (`tags: list = []`): Python refuses; use `field(default_factory=list)`.
2. **`Optional` without a check**: `x: int | None` used as an int. mypy catches it; the fix is an `if x is not None`.
3. **Inheriting where the subclass can't keep the parent's promises** (Liskov): it breaks callers who trust the parent.
4. **Catching `Exception` (or worse) around a whole function**: bugs become wrong answers instead of tracebacks.
5. **Logging and re-raising at every layer**: one problem, five tracebacks.
6. **`assert` on user input**: it vanishes under `-O`, and raises the wrong kind of error anyway.

## Practice

## Quiz

## Daily challenge

## Before you go

For each row of the "Describing data" table, write a one-line example of your own from memory, then check it. Tomorrow: the patterns from Weeks 35 and 36, and the layered architecture that holds them together.